# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [1]:
import numpy as np
import pandas as pd

# copied from week 6 example notebooks
def compute_qT_once(R, P, gamma, v):
    return R + gamma * P @ v

def iterate_values_once(R, P, gamma, v):
    return np.max(compute_qT_once(R, P, gamma, v), axis=0)

def value_iteration(R, P, gamma, max_iterations=10000, tolerance=1e-9):
    v_old = np.zeros(R.shape[-1])
    for i in range(max_iterations):
        v_new = iterate_values_once(R, P, gamma, v_old)
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return v_new
        v_old = v_new
    return v_old

def iterative_policy_evaluation(R, P, gamma, pi, max_iterations=10000, tolerance=1e-9):
    n = R.shape[-1]
    R_pi = R[pi, np.arange(n)].reshape(1, n)
    P_pi = P[pi, np.arange(n), :].reshape(1, n, n)
    v_old = np.zeros(n)
    for i in range(max_iterations):
        v_new = iterate_values_once(R_pi, P_pi, gamma, v_old)
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return v_new
        v_old = v_new
    return v_old

In [2]:
# Problem 1 Solve: 

transitions = pd.read_csv('twizzleflu-transitions.tsv', sep='\t')
rewards = pd.read_csv('twizzleflu-rewards.tsv', sep='\t')

states = ['exposed-1', 'exposed-2', 'exposed-3', 'symptoms-1', 'symptoms-2', 'symptoms-3', 'recovered']
actions = ['do-nothing', 'drink-oj', 'sleep-8']

n_states = len(states)
state_idx = {s: i for i, s in enumerate(states)}
action_idx = {a: i for i, a in enumerate(actions)}

R = np.zeros((len(actions), n_states))
for _, row in rewards.iterrows():
    R[action_idx[row['action']], state_idx[row['state']]] = row['reward']

P = np.zeros((len(actions), n_states, n_states))
for _, row in transitions.iterrows():
    P[action_idx[row['action']], state_idx[row['state']], state_idx[row['next_state']]] = row['probability']


# gamma=1 (no discounting) - value iteration still converges since recovered is absorbing
gamma = 1.0
pi_do_nothing = np.full(n_states, action_idx['do-nothing'], dtype='int64')
v_do_nothing = iterative_policy_evaluation(R, P, gamma, pi_do_nothing)

# rewards are negative, so flip sign to get discomfort
do_nothing_discomfort = -v_do_nothing
for s, val in zip(states, do_nothing_discomfort):
    print(f'{s}: {val:.4f}')


exposed-1: 3.4133
exposed-2: 4.2667
exposed-3: 5.3333
symptoms-1: 6.6667
symptoms-2: 5.0000
symptoms-3: 1.6667
recovered: -0.0000


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [3]:
pd.DataFrame({
    'state': states,
    'expected_discomfort': do_nothing_discomfort
}).to_csv('do-nothing-discomfort.tsv', sep='\t', index=False)


Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [4]:
v_opt = value_iteration(R, P, gamma)
pi_opt = np.argmax(compute_qT_once(R, P, gamma, v_opt), axis=0)

for s, a in zip(states, pi_opt):
    print(f'{s}: {actions[a]}')


exposed-1: sleep-8
exposed-2: sleep-8
exposed-3: sleep-8
symptoms-1: drink-oj
symptoms-2: drink-oj
symptoms-3: drink-oj
recovered: do-nothing


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [5]:
pd.DataFrame({
    'state': states,
    'action': [actions[a] for a in pi_opt]
}).to_csv('minimum-discomfort-actions.tsv', sep='\t', index=False)


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [6]:
v_comfort = iterative_policy_evaluation(R, P, gamma, pi_opt)
min_discomfort = -v_comfort

for s, val in zip(states, min_discomfort):
    print(f'{s}: {val:.4f}')


exposed-1: 0.7500
exposed-2: 1.5000
exposed-3: 3.0000
symptoms-1: 6.0000
symptoms-2: 4.5000
symptoms-3: 1.5000
recovered: -0.0000


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [7]:
pd.DataFrame({
    'state': states,
    'expected_discomfort': min_discomfort
}).to_csv('minimum-discomfort-values.tsv', sep='\t', index=False)


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [8]:
# -1 per day while symptomatic, 0 otherwise (exposed doesn't count)
sick_states = {'symptoms-1', 'symptoms-2', 'symptoms-3'}

R_dur = np.zeros_like(R)
for a in range(len(actions)):
    for i, s in enumerate(states):
        if s in sick_states:
            R_dur[a, i] = -1.0


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [9]:
rows = [{'action': actions[a], 'state': s, 'reward': R_dur[a, i]}
        for a in range(len(actions)) for i, s in enumerate(states)]

pd.DataFrame(rows).to_csv('duration-rewards.tsv', sep='\t', index=False)


Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [10]:
v_dur = value_iteration(R_dur, P, gamma)
pi_dur = np.argmax(compute_qT_once(R_dur, P, gamma, v_dur), axis=0)

for s, a in zip(states, pi_dur):
    print(f'{s}: {actions[a]}')


exposed-1: sleep-8
exposed-2: sleep-8
exposed-3: sleep-8
symptoms-1: do-nothing
symptoms-2: do-nothing
symptoms-3: do-nothing
recovered: do-nothing


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [11]:
pd.DataFrame({
    'state': states,
    'action': [actions[a] for a in pi_dur]
}).to_csv('minimum-duration-actions.tsv', sep='\t', index=False)


Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [12]:
v_dur_eval = iterative_policy_evaluation(R_dur, P, gamma, pi_dur)
sick_days = -v_dur_eval  # reward is -1/day so total reward = -(days sick)

for s, val in zip(states, sick_days):
    print(f'{s}: {val:.4f}')


exposed-1: 1.2500
exposed-2: 2.5000
exposed-3: 5.0000
symptoms-1: 10.0000
symptoms-2: 6.6667
symptoms-3: 3.3333
recovered: -0.0000


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [13]:
pd.DataFrame({
    'state': states,
    'expected_sick_days': sick_days
}).to_csv('minimum-duration-days.tsv', sep='\t', index=False)


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

In [14]:
v_speed = iterative_policy_evaluation(R, P, gamma, pi_dur)
speed_discomfort = -v_speed

for s, spd, opt in zip(states, speed_discomfort, min_discomfort):
    print(f'{s}: speed={spd:.4f}, minimize={opt:.4f}')


exposed-1: speed=0.8333, minimize=0.7500
exposed-2: speed=1.6667, minimize=1.5000
exposed-3: speed=3.3333, minimize=3.0000
symptoms-1: speed=6.6667, minimize=6.0000
symptoms-2: speed=5.0000, minimize=4.5000
symptoms-3: speed=1.6667, minimize=1.5000
recovered: speed=-0.0000, minimize=-0.0000


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [15]:
pd.DataFrame({
    'state': states,
    'speed_discomfort': speed_discomfort,
    'minimize_discomfort': min_discomfort
}).to_csv('policy-comparison.tsv', sep='\t', index=False)


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

### Acknowledgements: 

I relied upon the functions and examples from the three example notesbooks in dx704-examples/week06. I also have a code Linter in my VS Code instance for cleaning up my code.